To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [24]:
#%%capture
#import os

#!pip install pip3-autoremove
#!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
#!pip install unsloth
#!pip install transformers==4.56.2
#!pip install --no-deps trl==0.22.2

In [25]:
HF_TOKEN = 'hf_REDACTED_PUT_YOUR_TOKEN_IN_ENV'
!hf auth login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `paper` has been saved to C:\Users\progear\.cache\huggingface\stored_tokens
Your token has been saved to C:\Users\progear\.cache\huggingface\token
Login successful.
The current active token is: `paper`


In [26]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

==((====))==  Unsloth 2026.6.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5090 Laptop GPU. Num GPUs = 1. Max memory: 23.889 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [13]:
from unsloth import FastModel
from transformers import AutoModelForSequenceClassification
import torch

# Disable fast generation for bert!
#%env UNSLOTH_DISABLE_FAST_GENERATION = 1

import random
import numpy as np
import os

seed_value = 1998

# Python's random module
random.seed(seed_value)

# NumPy
np.random.seed(seed_value)

# PyTorch
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)  # For multi-GPU

# PyTorch backends (for deterministic behavior)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set Python hash seed (for string hashing)
os.environ['PYTHONHASHSEED'] = str(seed_value)

# Transformers library (if using Trainer)
from transformers import set_seed
set_seed(seed_value)

max_seq_length = 512 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


In [ ]:

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

id2label = {0: "OR", 1: "CG"} #, 2: "love", 3: "anger",4: "fear",5: "surprise"}

label2id = {"OR": 0, "CG": 1} #, "love": 2, "anger": 3, "fear": 4, "surprise": 5}

model, tokenizer = FastModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct", #"google/gemma-3-270m-it", #"LiquidAI/LFM2-350M", #"Pulk17/Fake-News-Detection", #"distilbert/distilbert-base-uncased-finetuned-sst-2-english",#"answerdotai/ModernBERT-large",
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = max_seq_length,
    dtype = dtype,
    num_labels  = 2,
    full_finetuning = False,
    id2label=id2label,
    label2id=label2id,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

We now add LoRA adapters so we only need to update a small amount of parameters!


In [28]:
#model.classifier = torch.nn.Linear(768, 2).to(model.device, dtype=torch.float32)

In [29]:
model = FastModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    #target_modules = #["query", "key", "value", "dense"], #["Wqkv", "Wo", "Wi"],
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # Checkpointing trades compute for VRAM; unnecessary on 24 GB (run uses ~6 GB)
    # and its CPU offloading was a bottleneck — re-enable "unsloth" if VRAM runs out.
    use_gradient_checkpointing = "unsloth", # True, False, or "unsloth" for our optimized version that trades compute for VRAM
    random_state = seed_value,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type="SEQ_CLS",
)

Unsloth: Allowing gradients for `base_model.model.score` since it's in `modules_to_save`.


<a name="Data"></a>
### Data Prep  
We now use the [Emotion dataset](https://huggingface.co/datasets/dair-ai/emotion) from `dair-ai`, which contains text labeled by emotion. In this example, we load the **unsplit** version and use only the first 30,000 samples.  

We then split the dataset into training (80%) and validation (20%), and apply tokenization to prepare the text for training.


In [14]:
from datasets import load_dataset

# Load the IMDB dataset
dataset = load_dataset('csv', data_files='dataset/fake reviews dataset.csv')
dataset

DatasetDict({
    train: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 40432
    })
})

In [15]:
"""max_len = max(len(tokenizer(text)['input_ids']) for text in dataset['train']['text_'])
print(max_len)"""

"max_len = max(len(tokenizer(text)['input_ids']) for text in dataset['train']['text_'])\nprint(max_len)"

In [16]:
"""import numpy as np

# Tokenize all texts and get their lengths
token_lengths = [len(tokenizer(text)['input_ids']) for text in dataset['train']['text_']]

# Calculate percentiles from 90 to 99
percentiles = np.percentile(token_lengths, range(90, 100))

# Display results
for i, p in enumerate(percentiles, start=90):
    print(f"{i}th percentile: {int(p)} tokens")

# Or get specific percentiles
p90 = np.percentile(token_lengths, 90)
p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)

print(f"\n90th percentile: {int(p90)}")
print(f"95th percentile: {int(p95)}")
print(f"99th percentile: {int(p99)}")"""

'import numpy as np\n\n# Tokenize all texts and get their lengths\ntoken_lengths = [len(tokenizer(text)[\'input_ids\']) for text in dataset[\'train\'][\'text_\']]\n\n# Calculate percentiles from 90 to 99\npercentiles = np.percentile(token_lengths, range(90, 100))\n\n# Display results\nfor i, p in enumerate(percentiles, start=90):\n    print(f"{i}th percentile: {int(p)} tokens")\n\n# Or get specific percentiles\np90 = np.percentile(token_lengths, 90)\np95 = np.percentile(token_lengths, 95)\np99 = np.percentile(token_lengths, 99)\n\nprint(f"\n90th percentile: {int(p90)}")\nprint(f"95th percentile: {int(p95)}")\nprint(f"99th percentile: {int(p99)}")'

In [17]:
# Split into training and validation sets
dataset = dataset['train'].train_test_split(test_size=0.2, seed=seed_value)
dataset_test = dataset['test'].train_test_split(test_size=0.7, seed=seed_value)

In [18]:
dataset_test

DatasetDict({
    train: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 2426
    })
    test: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 5661
    })
})

In [19]:
dataset_test['test'][1998]

{'category': 'Electronics_5',
 'rating': 1.0,
 'label': 'CG',
 'text_': 'not worth the money. not what I expected.Worst item in the world for the price.'}

In [20]:
from datasets import DatasetDict
dataset_train = DatasetDict({'train': dataset['train'], 'validation': dataset_test['train'], 'test': dataset_test['test']})

In [21]:
dataset_train

DatasetDict({
    train: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 32345
    })
    validation: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 2426
    })
    test: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 5661
    })
})

In [30]:
from datasets import Value

def tokenize_function(examples):
    return tokenizer_new(examples["text_"], padding=True, truncation=True)

def map_labels_to_ids(examples):
    examples["label"] = [label2id[label] for label in examples["label"]]
    return examples

# Apply the tokenizer to the dataset
train_dataset = dataset_train['train'].map(tokenize_function, batched=True)
val_dataset = dataset_train["validation"].map(tokenize_function, batched=True)
test_dataset = dataset_train["test"].map(tokenize_function, batched=True)

# Map string labels to integer IDs
train_dataset = train_dataset.map(map_labels_to_ids, batched=True)
val_dataset = val_dataset.map(map_labels_to_ids, batched=True)
test_dataset = test_dataset.map(map_labels_to_ids, batched=True)

# datasets 4.x keeps a modified column's original dtype, so the 0/1 ids above
# come back as the strings "0"/"1" — cast them to real ints for the collator
train_dataset = train_dataset.cast_column("label", Value("int64"))
val_dataset = val_dataset.cast_column("label", Value("int64"))
test_dataset = test_dataset.cast_column("label", Value("int64"))

Casting the dataset: 100%|██████████| 5661/5661 [00:00<00:00, 377355.38 examples/s]



We compute **class weights** using scikit-learn’s ```compute_class_weight```.  
This is useful when training on datasets where certain classes are underrepresented, ensuring the model does not become biased towards majority labels.

In [31]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)

In [32]:
# We rename the dataset column from **`label`** to **`labels`**, since this is the expected field name for Hugging Face `Trainer`.
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset = train_dataset.rename_column("text_", "text")
val_dataset = val_dataset.rename_column("text_", "text")
test_dataset = test_dataset.rename_column("text_", "text")

In [33]:
train_dataset

Dataset({
    features: ['category', 'rating', 'labels', 'text', 'input_ids', 'attention_mask'],
    num_rows: 32345
})

In [34]:
class_weights

array([1.00350583, 0.99651858])


We define a `compute_metrics` function to evaluate the model during training.  
Here we use **accuracy** from scikit-learn, which compares predicted labels with the ground truth.  

**[NOTE]** Accuracy is a good baseline, but for imbalanced datasets you may also want to track metrics like **F1-score**, **precision**, or **recall**.  


In [35]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    # Calculate confusion matrix
    cm = confusion_matrix(labels, preds)
    print("\nConfusion Matrix:\n")
    print(cm)

    metrics = {"accuracy": accuracy_score(labels, preds)}

    # Calculate precision, recall, f1-score for each class
    # The 'average=None' ensures scores are returned for each class
    # We use labels=np.unique(labels) to ensure metrics are calculated for all present labels
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=np.unique(labels))

    # Log metrics for each class, using id2label for descriptive names
    for i, label_name in id2label.items():
        metrics[f"precision_{label_name}"] = precision[i]
        metrics[f"recall_{label_name}"] = recall[i]
        metrics[f"f1_{label_name}"] = f1[i]

    return metrics

In [28]:
"""from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get predictions on the test dataset
predictions = trainer.predict(test_dataset)

# Extract predicted labels and true labels
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Generate classification report
report = classification_report(true_labels, predicted_labels, target_names=list(id2label.values()))
print("\nClassification Report:\n")
print(report)

# Generate confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)
print("\nConfusion Matrix:\n")
print(cm)"""

'from sklearn.metrics import classification_report, confusion_matrix\nimport numpy as np\n\n# Get predictions on the test dataset\npredictions = trainer.predict(test_dataset)\n\n# Extract predicted labels and true labels\npredicted_labels = np.argmax(predictions.predictions, axis=1)\ntrue_labels = predictions.label_ids\n\n# Generate classification report\nreport = classification_report(true_labels, predicted_labels, target_names=list(id2label.values()))\nprint("\nClassification Report:\n")\nprint(report)\n\n# Generate confusion matrix\ncm = confusion_matrix(true_labels, predicted_labels)\nprint("\nConfusion Matrix:\n")\nprint(cm)'

<a name="Train"></a>
### Train the model
Now let's use Huggingface  `Trainer`! More docs here: [Transformers docs](https://huggingface.co/docs/transformers/main_classes/trainer). We train for one full epoch (num_train_epochs=1) to get a meaningful result.

In [45]:
from transformers import TrainingArguments,Trainer,EarlyStoppingCallback
from unsloth import is_bfloat16_supported

trainer = Trainer(
    model = model,
    processing_class = tokenizer,
    eval_dataset = val_dataset,
    train_dataset = train_dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 32,  # was 32 x 2 accumulation — same effective batch of 64
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.06,
        num_train_epochs = 6,  # Increase from 2
        learning_rate = 5e-6,  # Reduce from 2e-5 for more stable training
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.03,  # Increase from 0.01 for stronger regularization
        eval_strategy = "steps",
        eval_steps = 100,  # was 50 — full eval pass every 50 steps dominated wall time
        max_grad_norm=1.0,
        label_smoothing_factor=0.01,
        lr_scheduler_type = "cosine_with_restarts",  # Change from linear for smoother decay
        lr_scheduler_kwargs = {"num_cycles": 2},
        seed = seed_value,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "steps",
        save_steps = 100,  # must stay a multiple of eval_steps for load_best_model_at_end
        load_best_model_at_end = True,
        metric_for_best_model = "eval_accuracy",
        greater_is_better = True,
        save_total_limit = 3,
    ),
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=8)],
)

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [46]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,345 | Num Epochs = 6 | Total steps = 3,036
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 35,194,624 of 529,229,184 (6.65% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Accuracy,Precision Or,Recall Or,F1 Or,Precision Cg,Recall Cg,F1 Cg
100,0.233300,0.278407,0.911789,0.946655,0.871048,0.907279,0.882576,0.951797,0.915881
200,0.193100,0.153016,0.955482,0.964346,0.945092,0.954622,0.947115,0.965686,0.956311
300,0.241500,0.120209,0.963314,0.939921,0.989185,0.963924,0.988803,0.937908,0.962683
400,0.062800,0.091263,0.976092,0.967320,0.985025,0.976092,0.985025,0.967320,0.976092
500,0.097400,0.091484,0.980214,0.971405,0.989185,0.980214,0.989185,0.971405,0.980214
600,0.111400,0.084910,0.983100,0.983347,0.982529,0.982938,0.982857,0.983660,0.983258
700,0.044700,0.079931,0.983100,0.976993,0.989185,0.983051,0.989247,0.977124,0.983148
800,0.034700,0.082274,0.985985,0.981848,0.990017,0.985915,0.990115,0.982026,0.986054
900,0.053200,0.074046,0.986810,0.982673,0.990849,0.986744,0.990939,0.982843,0.986874
1000,0.032900,0.074713,0.986810,0.978723,0.995008,0.986799,0.995017,0.978758,0.986820



Confusion Matrix:

[[1047  155]
 [  59 1165]]

Confusion Matrix:

[[1136   66]
 [  42 1182]]

Confusion Matrix:

[[1189   13]
 [  76 1148]]

Confusion Matrix:

[[1184   18]
 [  40 1184]]

Confusion Matrix:

[[1189   13]
 [  35 1189]]

Confusion Matrix:

[[1181   21]
 [  20 1204]]

Confusion Matrix:

[[1189   13]
 [  28 1196]]

Confusion Matrix:

[[1190   12]
 [  22 1202]]

Confusion Matrix:

[[1191   11]
 [  21 1203]]

Confusion Matrix:

[[1196    6]
 [  26 1198]]

Confusion Matrix:

[[1191   11]
 [  20 1204]]

Confusion Matrix:

[[1179   23]
 [  12 1212]]

Confusion Matrix:

[[1191   11]
 [  14 1210]]

Confusion Matrix:

[[1190   12]
 [  13 1211]]

Confusion Matrix:

[[1192   10]
 [  18 1206]]

Confusion Matrix:

[[1191   11]
 [  17 1207]]

Confusion Matrix:

[[1192   10]
 [  22 1202]]

Confusion Matrix:

[[1185   17]
 [  26 1198]]

Confusion Matrix:

[[1180   22]
 [  14 1210]]

Confusion Matrix:

[[1188   14]
 [  15 1209]]

Confusion Matrix:

[[1190   12]
 [  28 1196]]


<a name="Inference"></a>
### Inference
Let's run the model !

In [47]:
from transformers import pipeline

idx = 1998

sentence1 = test_dataset['text'][idx]

classifier = pipeline("sentiment-analysis", model=model,tokenizer=tokenizer)

print(id2label[test_dataset['labels'][idx]])
classifier(sentence1)

Device set to use cuda:0


CG


[{'label': 'CG', 'score': 0.9977038502693176}]

In [48]:
eval_results = trainer.evaluate(test_dataset)
print(eval_results)


Confusion Matrix:

[[2861   37]
 [  25 2738]]
{'eval_loss': 0.06913971155881882, 'eval_accuracy': 0.9890478714008126, 'eval_precision_OR': 0.9913374913374914, 'eval_recall_OR': 0.9872325741890959, 'eval_f1_OR': 0.9892807745504841, 'eval_precision_CG': 0.9866666666666667, 'eval_recall_CG': 0.9909518639160333, 'eval_f1_CG': 0.9888046226074395, 'eval_runtime': 53.5688, 'eval_samples_per_second': 105.677, 'eval_steps_per_second': 13.217, 'epoch': 4.150346191889218}


<a name="Save"></a>
### Saving finetuned models
To save the final model, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.


In [49]:
model.save_pretrained("model")  # Local saving
tokenizer.save_pretrained("model")
# model.push_to_hub("your_name/model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/model", token = "...") # Online saving

('model\\tokenizer_config.json',
 'model\\special_tokens_map.json',
 'model\\chat_template.jinja',
 'model\\vocab.json',
 'model\\merges.txt',
 'model\\added_tokens.json',
 'model\\tokenizer.json')

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>


In [38]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import torch
from unsloth import FastModel

id2label = {0: "OR", 1: "CG"}
label2id = {"OR": 0, "CG": 1}

model_new, tokenizer_new = FastModel.from_pretrained(
    model_name = "model",          # local folder with the saved LoRA adapter
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = 512,
    num_labels = 2,
    id2label = id2label,
    label2id = label2id,
    load_in_4bit = True,
    full_finetuning = False,
)
FastModel.for_inference(model_new)     # inference mode, no training overhead


# Move model to GPU if available
if torch.cuda.is_available():
    model_new = model_new.to("cuda")

# Define minimal TrainingArguments for evaluation
eval_args = TrainingArguments(
    output_dir="./eval_results_new_model", # New output directory for this model's results
    per_device_eval_batch_size=16, # Reduced batch size to prevent OOM
    report_to="none",
    # If memory still an issue, uncomment these for fp16/bf16 if your GPU supports it:
    # fp16 = True, # or bf16 = True
)

# Instantiate a new Trainer for evaluation with the new model
trainer_eval = Trainer(
    model=model_new,
    tokenizer=tokenizer_new,
    args=eval_args,
)

print("\n--- Running predictions on the test dataset ---\n")

# test_dataset's input_ids came from the Qwen tokenizer — those ids exceed
# ModernBERT's vocab and crash the GPU. Re-tokenize with ModernBERT's tokenizer.
test_dataset_bert = test_dataset.remove_columns(["input_ids", "attention_mask"])
test_dataset_bert = test_dataset_bert.map(
    lambda batch: tokenizer_new(batch["text"], truncation=True, max_length=512),
    batched=True,
)

predictions_new = trainer_eval.predict(test_dataset_bert)


# Extract predicted labels and true labels
predicted_labels_new = np.argmax(predictions_new.predictions, axis=1)
true_labels_new = predictions_new.label_ids

# Generate classification report
report_new = classification_report(true_labels_new, predicted_labels_new, target_names=list(id2label.values()), digits=3)
print("\nClassification Report:\n")
print(report_new)

# Generate confusion matrix
cm_new = confusion_matrix(true_labels_new, predicted_labels_new)
print("\nConfusion Matrix:\n")
print(cm_new)


==((====))==  Unsloth 2026.6.3: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GeForce RTX 5090 Laptop GPU. Num GPUs = 1. Max memory: 23.889 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unsloth: Allowing gradients for `base_model.model.score` since it's in `modules_to_save`.


d:\Dev\AI\Papers\Rs-QLoRA\.venv\Lib\site-packages\unsloth\models\_utils.py:2414: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  _original_trainer_init(self, *args, **kwargs)



--- Running predictions on the test dataset ---




Classification Report:

              precision    recall  f1-score   support

          OR      0.991     0.987     0.989      2898
          CG      0.986     0.991     0.988      2763

    accuracy                          0.989      5661
   macro avg      0.989     0.989     0.989      5661
weighted avg      0.989     0.989     0.989      5661


Confusion Matrix:

[[2859   39]
 [  25 2738]]
